# AML - SAIGE-GENE SET-BASED ANALYSIS - FIL NULL MODEL 
The following script is used to fit the null model for the set-based SAIGE-GENE analysis. The options useSparseGRMtoFitNULL, useSparseGRMforVarRatio, and isCateVarianceRatio must be set to TRUE. This script uses the sparse GRM, 2,000 randomly selected variants in the specified MAC categories, and the phenotype file with covariates to fit the null model.

In [7]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261024 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

In [8]:
import os
import tempfile
import subprocess

# Create a separate Cloud SDK configuration directory
os.environ["CLOUDSDK_CONFIG"] = tempfile.mkdtemp(
        prefix="gcloud-dsub-",
        dir="/tmp",
)

print("CLOUDSDK_CONFIG:", os.environ["CLOUDSDK_CONFIG"])
print("CLOUDSDK_CONFIG =", os.environ.get("CLOUDSDK_CONFIG", "(default)"))
print("HOME =", os.environ.get("HOME"))

subprocess.run(
        ["gcloud", "config", "get-value", "account"],
        check=False,
)

CLOUDSDK_CONFIG: /tmp/gcloud-dsub-cgza8q5s
CLOUDSDK_CONFIG = /tmp/gcloud-dsub-cgza8q5s
HOME = /home/dataproc
pet-27662581260322f686e2a@wb-fast-radish-6415.iam.gserviceaccount.com


CompletedProcess(args=['gcloud', 'config', 'get-value', 'account'], returncode=0)

# COMMAND 

In [35]:
%%writefile saige_gene_step01.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in aml_biosample_age; do 

            echo "Processing population: ${pop} | phenotype: ${pheno}"

            step1_fitNULLGLMM.R \
                --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
                --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
                --plinkFile="${variance_files}/${pop}_array.forCate_vr" \
                --phenoFile="${phenotype}/${pheno}_files/${pop}_phenotype_file.txt" \
                --SampleIDIncludeFile="${phenotype}/${pheno}_files/samplelist.txt" \
                --useSparseGRMtoFitNULL=TRUE \
                --useSparseGRMforVarRatio=TRUE \
                --IsOverwriteVarianceRatioFile=TRUE \
                --isCateVarianceRatio=TRUE \
                --phenoCol=aml_biosample_age \
                --traitType=binary \
                --covarColList="age,sex,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10" \
                --qCovarColList=sex \
                --sampleIDColinphenoFile=person_id \
                --outputPrefix="${output}/${pop}_${pheno}_case_subset_null_glmm" 

    done
done


Overwriting saige_gene_step01.sh


# SUBMIT COMMAND 

In [36]:
%%bash --out saige_gene_step01

source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 10 \
  --preemptible \
  --machine-type n1-standard-1 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --env phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files" \
  --env grm="${WORKSPACE_BUCKET}/data/saige_step00/grm" \
  --env variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files" \
  --env output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm" \
  --script "saige_gene_step01.sh"


Job properties:
  job-id: saige-gene--vienose--261025-171342-76
  job-name: saige-gene-step01
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/saige-gene--vienose--261025-171342-76-0-0
Launched job-id: saige-gene--vienose--261025-171342-76
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'saige-gene--vienose--261025-171342-76' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'saige-gene--vienose--261025-171342-76' --users 'vienose'


In [40]:
! dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'saige-gene--vienose--261025-171342-76' --users 'vienose' --status '*'

Job Name         Status    Last Update
---------------  --------  --------------
saige-gene-s...  SUCCESS   10-25 17:22:59



In [42]:
!gcloud storage ls ${WORKSPACE_BUCKET}/data/logging | grep 'saige-gene--vienose--261025-171342-76'

gs://rw-migration-aou-rw-9f879fc4/data/logging/saige-gene--vienose--261025-171342-76-stderr.log
gs://rw-migration-aou-rw-9f879fc4/data/logging/saige-gene--vienose--261025-171342-76-stdout.log
gs://rw-migration-aou-rw-9f879fc4/data/logging/saige-gene--vienose--261025-171342-76.log


In [44]:
!gcloud storage cat gs://rw-migration-aou-rw-9f879fc4/data/saige_phenotype_files/aml_biosample_age_files/EUR_phenotype_file.txt | wc -l 

169972


In [46]:
!gcloud storage cat gs://rw-migration-aou-rw-9f879fc4/data/saige_phenotype_files/aml_biosample_age_files/samplelist.txt | wc -l 

169802


In [43]:
!gcloud storage cat gs://rw-migration-aou-rw-9f879fc4/data/logging/saige-gene--vienose--261025-171342-76-stdout.log

Processing population: EUR
Processing population: EUR | phenotype: aml_biosample_age
R version 4.4.3 (2025-02-28)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 20.04.4 LTS

Matrix products: default
BLAS/LAPACK: /app/.pixi/envs/default/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: NA
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] optparse_1.7.5 SAIGE_1.5.0.2 

loaded via a namespace (and not attached):
 [1] R6_2.6.1           tidyselect_1.2.1   Matrix_1.7-3       lattice_0.22-7    
 [5] magrittr_2.0.3     splines_4.4.3      getopt_1.20.4      glue_1.8